In [ ]:
import pandas as pd
import matplotlib as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split


In [ ]:

file = 'Boston.csv'
df = pd.read_csv(file)
print(f'Head contents of {file} is \n{df.head(5)}')
columns = df.columns
print(f'Label of {file} : {columns}')
label_means = {
    'CRIME':'犯罪発生率',
    'ZN':'25,000平方フィート以上の住居区画の占める割合',
    'INDUS':'小売業以外の商業が占める面積の割合',
    'CHAS':'チャールズ川の近くか否か',
    'NOX':'窒素感化物の濃度',
    'RM':'住居の平均部屋数',
    'AGE':'1940年より前に建てられた物件の割合',
    'DIS':'ボストン市内の5つの雇用施設からの距離',
    'RAD':'環状高速道路へのアクセスしやすさ',
    'TAX':'10,000ドルあたりの不動産税率の総計',
    'PTRATIO':'町ごとの教員一人当たりの児童生徒数',
    'LSTAT':'人口における低所得者の割合',
    'PRICE':'その地域の住宅平均価格'
    }
print('ラベルの意味：')
for key, value in label_means.items():
    print(key,':',value)
print(f'Matrix size : {df.shape}')

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

# plt.figure(figsize=(5,3))
# plt.xlabel('Value')
# plt.ylabel('Frequency')

labels = [ 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'LSTAT', 'PRICE']
for label in labels:
    print(f'{label}:{label_means[label]}')
    df[label].plot(kind='hist',xlabel=label)
    plt.show()
    plt.close()
    



In [ ]:
# ユニークな値を調べる
import matplotlib.pyplot as plt
%matplotlib inline

for i in {0,3}:
    print(f'{df.columns[i]}:{label_means[df.columns[i]]}')
    print(f'Unique item of {df.columns[i]} = {df[df.columns[i]].unique()}')
    value_counts = df[df.columns[i]].value_counts()
    print(f'Each count of unique item of {df.columns[i]}:{value_counts}')

    value_counts.plot(kind='bar')
    plt.show()
    # plt.close()

In [ ]:
# 文字列データのCRIMEのダミー変数を作る
crime_flsgs = pd.get_dummies(df['CRIME'], drop_first = True, dtype = int); print(crime_flsgs.head(5))
# ダミー変数を元のデータフレームに列接続する
df2 = pd.concat([df,crime_flsgs], axis = 1); print(df2.head(5))
# CREIME列を削除する
df2 = df2.drop('CRIME', axis=1); print(df2.head(5))
label_means['low'] = '犯罪の発生率が低いか否か'
label_means['very_low'] = '犯罪の発生率が非常に低いか否か'
print('ラベルの意味：')
for key, value in label_means.items():
    print(key,':',value)

In [ ]:
# ホールドアウト法により訓練データとテストデータに分割する

# test_size　: 検証に利用する割合、注意：整数値を指定するとテストデータ件数とみなされる
train_val, test = train_test_split(df2,test_size=0.2, random_state=0)

print("訓練データおよび検証用データの特徴量の行列サイズ：",train_val.shape)
print(train_val.head(3), "\n")
print("テストデータの特徴量の行列サイズ：",test.shape)
print(test.head(3),"\n")


In [ ]:
print('欠損値の数を確認\n',train_val.isnull().sum())

In [ ]:
train_val_mean = train_val.mean()
print('各列の平均値：\n',train_val_mean)
# 欠損値を平均値で穴埋め
train_val2 = train_val.fillna(train_val_mean)
print('\n欠損値の数を確認:\n',train_val2.isnull().sum())

In [ ]:
# 散布図をプロットして外れ値が無いか確認する
colname = train_val2.columns
for name in colname:
    print(f'{name}:{label_means[name]}')
    train_val2.plot(kind='scatter', x = name, y = 'PRICE')
    plt.show()

In [ ]:
#外れ値があるインデックスを確認する

label = 'RM'
out_line = train_val2[(train_val2[label] < 6) & (train_val2['PRICE']>40)].index
print(f'{label}の外れ値のインデックス：{out_line}')

label = 'PTRATIO'
out_line2 = train_val2[(train_val2[label] > 18) & (train_val2['PRICE']>40)].index
print(f'{label}の外れ値のインデックス：{out_line}')


In [ ]:
# 外れ値を削除する
train_val3 = train_val2.drop([76], axis = 0)
train_val3.shape

In [ ]:
# 絞り込んだ列以外を取り除く
col = ['INDUS', 'NOX', 'RM', 'PTRATIO', 'LSTAT', 'PRICE']

train_val4= train_val3[col]
print(train_val4.shape)
train_val4.head(5)

In [ ]:
# 列同士の相関係数を調べる
train_val4.corr()

In [ ]:
# 各列とPRICE列との相関係数を見る
train_cor = train_val4.corr()['PRICE']
train_cor

In [ ]:
# mapメソッドで要素に関数を適用する
se = pd.Series([1,-2,3,-4])
print('abs:\n',se.map(abs))
print('x*2:\n',se.map(lambda x: x*2))
print('x^2:\n',se.map(lambda x: x*x))

In [ ]:
# 相関行列のPRICE列との相関係数を絶対値に変換する
abs_cor = train_cor.map(abs)
abs_cor

In [ ]:
# sort_valuesメソッドで要素を降順に並べ変える
abs_cor.sort_values(ascending=False)

In [ ]:
# PRICEと相関の強い特徴量とする
col = ['RM', 'LSTAT', 'PTRATIO']
x = train_val4[col]
t = train_val4[['PRICE']]

# 訓練データと検証データに分割する
x_train, x_val, y_train, y_val = train_test_split(x,t, test_size=0.2, random_state = 0)
print(f'x_trainの行列サイズ：\n{x_train.shape}')
print(f'x_val.shapeの行列サイズ:\n{x_val.shape}')

In [ ]:
# 特徴量を標準化するためにscikit-learnのpreprocessingモジュールを使う
from sklearn.preprocessing import StandardScaler

sc_model_x = StandardScaler()
sc_model_x.fit(x_train)

# 各列のデータを標準化してsx_xに代入
sc_x = sc_model_x.transform(x_train)
sc_x

In [ ]:
# 標準化された特徴量の平均が0、標準偏差が1になっているか確認
# array型だと見づらいのでデータフレームに変換
tmp_df = pd.DataFrame(sc_x, columns = x_train.columns)
print(f'平均値：\n{tmp_df.mean()}')
print(f'標準偏差：\n{tmp_df.std()}')

In [ ]:
# 正解データを標準化する
sc_model_y = StandardScaler()
sc_model_y.fit(y_train)

sc_y = sc_model_y.transform(y_train)

In [ ]:
# 標準化された正解データの平均が0、標準偏差が1になっているか確認
# array型だと見づらいのでデータフレームに変換
tmp_df = pd.DataFrame(sc_y, columns = y_train.columns)
print(f'平均値：\n{tmp_df.mean()}')
print(f'標準偏差：\n{tmp_df.std()}')

In [ ]:
# 標準化したデータで学習させる
model = LinearRegression()
model.fit(sc_x,sc_y)
model.score(x_val,y_val)

In [ ]:
# 検証データも標準化する必要がある。検証データを訓練データの平均値と標準偏差を用いて標準化する
sc_x_val = sc_model_x.transform(x_val)
sc_y_val = sc_model_y.transform(y_val)

# 標準化した検証データで決定係数を計算
model.score(sc_x_val,sc_y_val)

In [ ]:
# チューニングのためのlearn関数の定義
def learn(x, t):
    # 訓練データと検証データに分割する
    x_train, x_val, y_train, y_val = train_test_split(x,t, test_size=0.2, random_state = 0)
    # 訓練データを標準化
    sc_model_x = StandardScaler()
    sc_model_x.fit(x_train)
    sc_x_train = sc_model_x.transform(x_train)
    sc_model_y = StandardScaler()
    sc_model_y.fit(y_train)
    sc_y_train = sc_model_y.transform(y_train)
    # 標準化したデータで学習させる
    model = LinearRegression()
    model.fit(sc_x_train,sc_y_train)

    # 検証データを標準化
    sc_x_val = sc_model_x.transform(x_val)
    sc_y_val = sc_model_y.transform(y_val)
    # 訓練データと検証データの決定係数計算
    train_score = model.score(sc_x_train,sc_y_train)
    val_score = model.score(sc_x_val,sc_y_val)

    return train_score, val_score

In [ ]:
# Learn関数を実行する
x = train_val3.loc[ : , ['RM','LSTAT','PTRATIO']]
t = train_val3[ ['PRICE']]

s1, s2 = learn(x,t)
print(s1,s2)

In [ ]:
# 特徴量にINDUSを追加してLearn関数を実行する
x = train_val3.loc[ : , ['RM','LSTAT','PTRATIO','INDUS']]
t = train_val3[ ['PRICE']]

s1, s2 = learn(x,t)
print(s1,s2)

In [ ]:
#INDUSを外す
x = x.drop('INDUS',axis=1)
print(x.head(3))

# RMの2乗の列を追加する
x['RM2'] = x['RM']**2
print(x.head(3))

s1, s2 = learn(x,t)
print(f'決定係数：訓練データ{s1},検証データ{s2}')

In [ ]:
# RM,RM2とPRICEの相関関係を散布図で確認する
df_plot = pd.DataFrame()
df_plot['RM'] = x['RM']
df_plot['PRICE'] = t['PRICE']
df_plot['RM2'] = x['RM2']
df_plot.head(5)
df_plot.plot(kind='scatter', x = 'RM', y = 'PRICE')
plt.show()
df_plot.plot(kind='scatter', x = 'RM2', y = 'PRICE')
plt.show()


In [ ]:
# LSTAT列とPTRATIO列で新しい列を特徴量に追加する
# x['LSTAT_INV'] = 1/x['LSTAT']
# #LSTATを外す
# x = x.drop('LSTAT',axis=1)
# print(x.head(3))
x['LSTAT2'] = x['LSTAT']**2
print(x.head(3))
s1, s2 = learn(x,t)
print(f'決定係数：訓練データ{s1},検証データ{s2}')

x['PTRATIO2'] = x['PTRATIO']**2
print(x.head(3))
s1, s2 = learn(x,t)
print(f'決定係数：訓練データ{s1},検証データ{s2}')

In [ ]:
# LSTAT,LSTAT_INVとPRICEの相関関係を散布図で確認する
df_plot = pd.DataFrame()
df_plot['LSTAT'] = x['LSTAT']
df_plot['PRICE'] = t['PRICE']
# df_plot['LSTAT_INV'] = x['LSTAT_INV']
df_plot['LSTAT2'] = x['LSTAT2']
print(df_plot.head(5))
df_plot.plot(kind='scatter', x = 'LSTAT', y = 'PRICE')
plt.show()
# df_plot.plot(kind='scatter', x = 'LSTAT_INV', y = 'PRICE')
df_plot.plot(kind='scatter', x = 'LSTAT2', y = 'PRICE')
plt.show()


In [ ]:
# 交互作用特徴量を追加する
x['RM*LSTAT'] = x['RM']*x['LSTAT']
print(x.head(3))

s1, s2 = learn(x,t)
print(f'決定係数：訓練データ{s1},検証データ{s2}')

In [ ]:
# 訓練データと検証データを合わせて再学習させるので、再度標準化する
sc_model_x2 = StandardScaler()
sc_model_x2.fit(x)
sc_x = sc_model_x2.transform(x)

sc_model_y2 = StandardScaler()
sc_model_y2.fit(t)
sc_y = sc_model_y2.transform(t)
model = LinearRegression()
model.fit(sc_x, sc_y)


In [ ]:
# テストデータの前処理
test2 = test.fillna(train_val.mean())
print(test2.isnull().sum())
x_test = test2.loc[:, ['RM','LSTAT','PTRATIO']]; print(x_test.shape)
print(x_test.head(3))
y_test = test2[['PRICE']]; print(y_test.shape)
print(y_test.head(3))

x_test['RM2'] = x_test['RM']**2
x_test['LSTAT2'] = x_test['LSTAT']**2
x_test['PTRATIO2'] = x_test['PTRATIO']**2
x_test['RM*LSTAT'] = x_test['RM']*x_test['LSTAT']
print(x_test.head(3))

sc_x_test = sc_model_x2.transform(x_test); print(sc_x_test)
sc_y_test = sc_model_y2.transform(y_test); print(sc_y_test)

model.score(sc_x_test, sc_y_test)

In [ ]:
# モデルの保存(標準化モデル込み)
import pickle
with open('boston.pkl', 'wb') as f:
    pickle.dump(model, f)
with open('boston_scx.pkl', 'wb') as f:
    pickle.dump(sc_model_x2, f)
with open('boston_scy.pkl', 'wb') as f:
    pickle.dump(sc_model_y2, f)